<a href="https://colab.research.google.com/github/VictorCamargo344/Aurora-Fase3/blob/main/script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Missão Aurora Siger | Colônia de Marte
### Fase 3 — Análise de Dados, Previsão e Decisão Automatizada

# ETAPA 1 — Organização dos Dados da Colônia



In [1]:
import copy

# Estrutura hierárquica dos dados da colônia (organização em chave-valor aninhado)
# Sistema energético → solar / eólico / reserva
# Sistema ambiental  → temperatura, vento, pressão
# Sistema operacional → módulos pousados na Fase 2

dados_colonia = {
    "nome": "Aurora Siger",
    "localizacao": "Marte — Hellas Planitia",
    "sistemas": {
        "energetico": {
            "solar": {
                "paineis": 120,
                "geracao_atual_kwh": 45.0,
                "eficiencia_pct": 78,
                "status": "ativo"
            },
            "eolico": {
                "turbinas": 4,
                "geracao_atual_kwh": 18.0,
                "velocidade_vento_kmh": 12.0,
                "status": "ativo"
            },
            "reserva": {
                "nivel_kwh": 320.0,
                "capacidade_kwh": 500.0,
                "percentual_pct": 64.0
            }
        },
        "ambiental": {
            "temperatura_interna_c": 22.0,
            "temperatura_externa_c": -63.0,
            "velocidade_vento_kmh": 12.0,
            "pressao_pa": 636.0,
            "radiacao_wm2": 590.0
        },
        "operacional": {
            "modulos": [
                {"nome": "Habitat Alpha", "tipo": "Habitacao",   "consumo_kwh": 30.0, "prioridade": 1, "status": "ativo"},
                {"nome": "MedBase",       "tipo": "Medicina",    "consumo_kwh": 20.0, "prioridade": 2, "status": "ativo"},
                {"nome": "Reator Solar",  "tipo": "Energia",     "consumo_kwh": 15.0, "prioridade": 2, "status": "ativo"},
                {"nome": "BioLab",        "tipo": "Laboratorio", "consumo_kwh": 10.0, "prioridade": 3, "status": "ativo"},
                {"nome": "LogiStore",     "tipo": "Logistica",   "consumo_kwh":  8.0, "prioridade": 4, "status": "ativo"},
                {"nome": "TerraBot",      "tipo": "Mineracao",   "consumo_kwh": 12.0, "prioridade": 5, "status": "ativo"}
            ]
        }
    }
}

print("Estrutura de dados da colonia carregada com sucesso.")

Estrutura de dados da colonia carregada com sucesso.


In [ ]:
# Exibe a hierarquia de sistemas da colonia de forma legivel
def exibir_hierarquia(dados, nivel=0):
    indent = "    " * nivel
    for chave, valor in dados.items():
        if isinstance(valor, dict):
            print(f"{indent}[{chave}]")
            exibir_hierarquia(valor, nivel + 1)
        elif isinstance(valor, list):
            print(f"{indent}[{chave}]")
            for item in valor:
                if isinstance(item, dict):
                    nome = item.get("nome", str(item))
                    consumo = item.get("consumo_kwh", "")
                    status = item.get("status", "")
                    info = f" | consumo: {consumo} kWh | status: {status}" if consumo != "" else ""
                    print(f"{indent}    - {nome}{info}")
        else:
            print(f"{indent}  {chave}: {valor}")

print("Estrutura Hierarquica da Colonia Aurora Siger")
print("=" * 55)
exibir_hierarquia(dados_colonia)

In [ ]:
# Resumo dos sensores em formato chave-valor para acesso rapido
def obter_resumo_sensores(dados):
    sis = dados["sistemas"] ## Variável criada apenas para evitar muita repetição de texto
    ## extração dos valores que interessam, guardando cada um em uma variável local.
    solar   = sis["energetico"]["solar"]["geracao_atual_kwh"]
    eolico  = sis["energetico"]["eolico"]["geracao_atual_kwh"]
    reserva = sis["energetico"]["reserva"]["percentual_pct"]
    t_int   = sis["ambiental"]["temperatura_interna_c"]
    t_ext   = sis["ambiental"]["temperatura_externa_c"]
    vento   = sis["ambiental"]["velocidade_vento_kmh"]

    ## Criando um novo dicionário com os dados resumidos
    return {
        "geracao_solar_kwh":    solar,
        "geracao_eolica_kwh":   eolico,
        "geracao_total_kwh":    solar + eolico,
        "reserva_pct":          reserva,
        "temperatura_interna_c": t_int,
        "temperatura_externa_c": t_ext,
        "velocidade_vento_kmh": vento
    }

sensores = obter_resumo_sensores(dados_colonia)

print("Leitura Atual dos Sensores (chave-valor):")
print("-" * 40)
for chave, valor in sensores.items():
    print(f"  {chave}: {valor}")

# ETAPA 2 — Cálculo da Geração Total e do Consumo de Energia

In [10]:
# Calcula a geracao total somando solar + eolico
def calcular_geracao_total(dados):
    sis = dados["sistemas"]["energetico"]
    return sis["solar"]["geracao_atual_kwh"] + sis["eolico"]["geracao_atual_kwh"]

# Calcula o consumo total somando todos os modulos ativos
def calcular_consumo_total(dados):
    modulos = dados["sistemas"]["operacional"]["modulos"]
    return sum(m["consumo_kwh"] for m in modulos if m["status"] == "ativo")

# Variaveis globais com os valores atuais da colonia (usadas nas etapas seguintes)
geracao     = calcular_geracao_total(dados_colonia)
consumo     = calcular_consumo_total(dados_colonia)
reserva_pct = dados_colonia["sistemas"]["energetico"]["reserva"]["percentual_pct"]

print(f"Geracao atual: {geracao} kWh | Consumo atual: {consumo} kWh | Reserva: {reserva_pct}%")

Geracao atual: 63.0 kWh | Consumo atual: 95.0 kWh | Reserva: 64.0%


# ETAPA 3 — Regras de Decisão (Lógica do Sistema)

In [5]:
# Ordena modulos por prioridade (menor numero = mais urgente / suporte a vida primeiro)
# Essa função é idêntica à ordenar_por_prioridade do MGPEB
def priorizar_modulos(modulos):
    lista = list(modulos)
    n = len(lista)
    for i in range(n):
        for j in range(0, n - i - 1):
            if lista[j]["prioridade"] > lista[j + 1]["prioridade"]:
                lista[j], lista[j + 1] = lista[j + 1], lista[j]
    return lista

# Motor de decisao unico: avalia o estado da colonia, exibe diagnostico e retorna status + alertas + acoes
def tomar_decisao(geracao, consumo, reserva_pct, temp_ext=0, vento=0):
    alertas = []
    acoes   = []
    saldo   = geracao - consumo

    # Cabecalho do diagnostico
    print("Analise de Energia")
    print("-" * 40)
    print(f"  Geracao total:  {geracao:.1f} kWh eq.")
    print(f"  Consumo total:  {consumo:.1f} kWh eq.")
    print(f"  Saldo:          {saldo:+.1f} kWh eq.")
    print(f"  Reserva:        {reserva_pct:.1f}%")
    print()

    # Regra 1: CRITICO — consumo > geracao E reserva < 20%
    if consumo > geracao and reserva_pct < 20:
        status = "CRITICO"
        alertas.append("Consumo excede geracao com reserva critica (< 20%)")
        acoes.append("Desligar TerraBot e LogiStore imediatamente")
        acoes.append("Manter apenas Habitat Alpha, MedBase e Reator Solar")

    # Regra 2: ALERTA — consumo > geracao
    elif consumo > geracao:
        status = "ALERTA"
        alertas.append(f"Consumo ({consumo:.0f} kWh) maior que geracao ({geracao:.0f} kWh)")
        acoes.append("Ativar modo economia")
        acoes.append("Reduzir consumo de TerraBot e LogiStore")

    # Regra 3: ATENCAO — reserva baixa E consumo elevado
    elif reserva_pct < 50 and consumo > geracao * 0.80:
        status = "ATENCAO"
        alertas.append(f"Reserva abaixo de 50% ({reserva_pct:.0f}%) com consumo elevado")
        acoes.append("Reduzir operacao de BioLab e LogiStore")

    # Regra 4: EFICIENTE — geracao muito acima do consumo
    elif geracao > consumo * 1.5:
        status = "EFICIENTE"
        alertas.append(f"Geracao excedente: {abs(saldo):.1f} kWh acima do consumo")
        acoes.append("Armazenar energia excedente nas baterias")

    # Regra 5: NORMAL
    else:
        status = "NORMAL"
        acoes.append("Manter operacao atual — sistema estavel")

    # Regra extra: temperatura externa critica
    if temp_ext < -150:
        alertas.append(f"Temperatura externa critica: {temp_ext}C")
        acoes.append("Verificar isolamento dos modulos externos")

    # Regra extra: vento forte (risco estrutural)
    if vento > 80:
        alertas.append(f"Vento acima do limite seguro: {vento} km/h")
        acoes.append("Recolher paineis solares externos")
    elif vento > 40:
        acoes.append(f"Vento favoravel ({vento} km/h) — aumentar geracao eolica")

    # Exibe status, alertas e acoes
    print(f"STATUS: {status}")
    if alertas:
        print("Alertas:")
        for a in alertas:
            print(f"  -> {a}")
    print("Acoes recomendadas:")
    for a in acoes:
        print(f"  * {a}")

    return status, alertas, acoes

In [ ]:
### Bloco de validação das Regras

## Teste com o cenário atual da colônia
print("TESTE 1 — Cenario atual da colonia")
print("=" * 55)
reserva = dados_colonia["sistemas"]["energetico"]["reserva"]["percentual_pct"]
temp_ext = dados_colonia["sistemas"]["ambiental"]["temperatura_externa_c"]
vento = dados_colonia["sistemas"]["ambiental"]["velocidade_vento_kmh"]
tomar_decisao(geracao, consumo, reserva, temp_ext, vento)
print()

# --- Teste 2: CRITICO (consumo > geracao E reserva < 20%) ---
print("=" * 55)
print("TESTE 2 — Cenario CRITICO")
print("=" * 55)
tomar_decisao(geracao=30, consumo=80, reserva_pct=15, temp_ext=-60, vento=10)
print()


# --- Teste 3: ALERTA (consumo > geracao, mas reserva ok) ---
print("=" * 55)
print("TESTE 3 — Cenario ALERTA")
print("=" * 55)
tomar_decisao(geracao=40, consumo=70, reserva_pct=60, temp_ext=-50, vento=15)
print()


# --- Teste 4: ATENCAO (reserva < 50% E consumo proximo da geracao) ---
print("=" * 55)
print("TESTE 4 — Cenario ATENCAO")
print("=" * 55)
tomar_decisao(geracao=60, consumo=55, reserva_pct=40, temp_ext=-55, vento=20)
print()


# --- Teste 5: EFICIENTE (geracao muito acima do consumo) ---
print("=" * 55)
print("TESTE 5 — Cenario EFICIENTE")
print("=" * 55)
tomar_decisao(geracao=150, consumo=70, reserva_pct=80, temp_ext=-40, vento=25)
print()


# --- Teste 6: NORMAL (sistema equilibrado) ---
print("=" * 55)
print("TESTE 6 — Cenario NORMAL")
print("=" * 55)
tomar_decisao(geracao=80, consumo=70, reserva_pct=70, temp_ext=-50, vento=15)
print()


# --- Teste 7: Regras extras — temperatura critica + vento forte ---
print("=" * 55)
print("TESTE 7 — Regras extras (temperatura + vento)")
print("=" * 55)
tomar_decisao(geracao=80, consumo=70, reserva_pct=70, temp_ext=-160, vento=90)
print()


# --- Teste 8: Vento favoravel (entre 40 e 80) ---
print("=" * 55)
print("TESTE 8 — Vento favoravel para geracao eolica")
print("=" * 55)
tomar_decisao(geracao=80, consumo=70, reserva_pct=70, temp_ext=-50, vento=50)

# ETAPA 4 — Previsão por Regressão Linear

In [13]:
# Dados historicos coletados pela colonia:
# velocidade do vento (km/h) x energia eolica gerada (kWh eq.)

historico_vento_kmh   = [4,  6,  8,  10, 12, 14, 16, 18, 20]
historico_energia_kwh = [10, 15, 20, 25, 30, 35, 40, 45, 50]

print("Historico de Vento x Energia Eolica:")
print("-" * 40)
print(f"  Vento  (km/h): {historico_vento_kmh}")
print(f"  Energia (kWh): {historico_energia_kwh}")

Historico de Vento x Energia Eolica:
----------------------------------------
  Vento  (km/h): [4, 6, 8, 10, 12, 14, 16, 18, 20]
  Energia (kWh): [10, 15, 20, 25, 30, 35, 40, 45, 50]


In [14]:
# Regressao linear simples — metodo dos minimos quadrados
# Calcula os coeficientes m (inclinacao) e b (intercepto) da reta y = m*x + b
def calcular_regressao_linear(x, y):
    n = len(x)
    soma_x  = sum(x)
    soma_y  = sum(y)
    soma_xy = sum(x[i] * y[i] for i in range(n))
    soma_x2 = sum(xi ** 2 for xi in x)

    m = (n * soma_xy - soma_x * soma_y) / (n * soma_x2 - soma_x ** 2)
    b = (soma_y - m * soma_x) / n

    return round(m, 4), round(b, 4)

# Usa a reta ajustada para estimar energia dado um vento novo
def prever_energia_eolica(velocidade_kmh, m, b):
    return round(m * velocidade_kmh + b, 2)

# Ajusta o modelo com os dados historicos
m_reg, b_reg = calcular_regressao_linear(historico_vento_kmh, historico_energia_kwh)

print("Modelo de Regressao Linear Ajustado:")
print("-" * 40)
print(f"  Formula: Energia = {m_reg} * Vento + {b_reg}")
print(f"  Coeficiente angular (m): {m_reg}")
print(f"  Intercepto (b):          {b_reg}")

Modelo de Regressao Linear Ajustado:
----------------------------------------
  Formula: Energia = 2.5 * Vento + 0.0
  Coeficiente angular (m): 2.5
  Intercepto (b):          0.0
